# Data Science Programming - Sesion 4 (via avanzada)

**Especializacion en Ciencia de Datos - Universidad Santo Tomas - Tunja**
**Prof. Hugo Hernandez | Martes 15 de septiembre, 2026**

---

## Antes de empezar, lee esto

Este notebook **reemplaza** al de la sesion. Si ya programas, trabaja aqui a tu ritmo
mientras el resto del grupo ve los fundamentos de limpieza.

- **No vale nota.** Existe para que no te toque esperar.
- **Trabajas solo.** Durante la clase estare con quienes estan empezando, asi que este
  notebook trae todo lo que necesitas. Cada reto se verifica solo: corres la celda de
  verificacion y sabes si lo lograste.
- **No tienes que entregarlo.** Estos retos no se califican. Subelo al aula solo si quieres que lo revise y te comente por escrito: es totalmente opcional y no hay que exponer.
- Si un reto se te atraviesa, sigue con el siguiente y vuelve despues.

Cubre el mismo temario que la clase normal, asi que no te quedas por fuera de nada que
necesites para la **Actividad 2**. Solo que llegas mas lejos: la clase normal escribe
cinco funciones de limpieza; aqui vas a escribir las herramientas que las orquestan.

---
## Repaso relampago

Todo lo que la clase normal ve en la primera hora, en dos tablas. Si algo de esto no te
suena, mejor toma el notebook normal.

| Problema | Deteccion | Correccion tipica |
|---|---|---|
| Faltantes | `df.isna().sum()` | `fillna(media/mediana)`, `KNNImputer`, o `dropna` |
| Duplicados exactos | `df.duplicated().sum()` | `drop_duplicates(keep='first')` |
| Duplicados parciales | `df.duplicated(subset=['id'])` | decidir cual fila sobrevive |
| Texto inconsistente | `df[col].unique()` | `.str.strip().str.lower().str.title()` |
| Fuera de rango | mascara `(s < min) \| (s > max)` | a `NaN` y despues imputar, o descartar fila |
| Outliers | Z-Score o IQR | recortar, marcar, o dejar y documentar |
| Escalas distintas | `df.describe()` | `MinMaxScaler`, `StandardScaler` |

| Criterio de outlier | Formula | Cuando usarlo |
|---|---|---|
| Z-Score | `abs(x - media) / desviacion > 3` | distribucion aproximadamente normal |
| IQR (Tukey) | fuera de `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` | distribucion sesgada; no lo mueve el extremo |

**El orden importa:** duplicados, texto, rangos, outliers, faltantes, escalar. Si imputas
antes de sacar el valor absurdo, el absurdo se cuela en la media con la que rellenas.

### Cuatro piezas que la clase normal no alcanza a ver y que aqui vas a usar

| Pieza | Para que |
|---|---|
| `unicodedata.normalize('NFKD', s)` | separa la letra de su tilde; despues filtras las marcas |
| `df.groupby(col)[c].transform(f)` | devuelve una Serie **alineada con el indice original**, no una tabla agregada |
| `pd.api.types.is_numeric_dtype(s)` | preguntar el tipo sin adivinar por el nombre de la columna |
| `Series.fillna(otra_serie)` | rellenar hueco por hueco desde otra Serie, emparejando por indice |

Y un detalle que muerde: `s.isna()` propaga `NaN` en las comparaciones. `(s > 5)` sobre una
columna con huecos devuelve `False` en esas posiciones en pandas moderno, pero **si escribes
`~(s <= 5)` obtienes `True`**. No son lo mismo. Cuando el resultado es una mascara booleana,
cierra siempre con `.fillna(False)` y di en el contrato que un hueco no es una violacion.

---
## La tabla sucia

Los cuatro retos giran alrededor de la misma tabla: 28 ventas de una tienda de barrio en
Boyaca. La construimos a mano y no la bajamos de internet por una razon: **sabemos
exactamente que defectos tiene**, asi que los asserts pueden comprobar si los corregiste.

| Defecto sembrado | Donde |
|---|---|
| Duplicado exacto | una fila repetida letra por letra |
| Duplicado parcial | dos filas del mismo `id_venta` que **solo son iguales despues de normalizar el texto** |
| Faltantes | 3 huecos en `precio`, 2 en `calificacion` |
| Texto inconsistente | mayusculas, espacios sobrantes, espacio doble interno, tildes y enies |
| Variante de categoria | una ciudad escrita como `'Bogota  D.C.'` |
| Outlier claro | un precio de 980000 entre precios de 8500 a 26000 |
| Outlier ambiguo | un precio de 45000: IQR lo marca, Z-Score no. Ese es el punto |
| Fuera de rango imposible | una calificacion de 9 en escala 1 a 5, y unas unidades en -2 |

Las tildes y enies van escritas como escapes `\u00e1`, `\u00f1`. Es el mismo caracter, pero
el archivo queda en ASCII puro y no depende de la codificacion del editor. En un CSV real te
vas a encontrar el problema al reves: acuerdate de `pd.read_csv(..., encoding='utf-8')`.

Ejecuta esta celda antes de empezar.

In [ ]:
import pandas as pd
import numpy as np
import unicodedata

ventas = pd.DataFrame({
    'id_venta': [301, 302, 303, 304, 305, 306, 307, 308, 309, 310,
                 311, 312, 313, 314, 315, 316, 317, 318, 319, 320,
                 321, 322, 323, 324, 325, 326, 320, 314],
    'vendedor': ['Rojas', 'alvarez', 'NINO', ' Rojas ', 'Suarez',
                 'rojas', '\u00c1lvarez', 'ni\u00f1o', 'SU\u00c1REZ', 'Rojas',
                 'alvarez', 'Ni\u00f1o', 'suarez', 'ROJAS', '\u00c1lvarez',
                 'nino ', 'Suarez', 'rojas', 'Alvarez', 'Suarez',
                 'Ni\u00f1o', 'Rojas', 'alvarez', 'NI\u00d1O', 'suarez',
                 'Rojas', 'Suarez', 'Rojas'],
    'ciudad':   ['Tunja', 'bogota', 'TUNJA', 'Tunja ', 'Duitama',
                 'tunja', 'Bogota  D.C.', 'duitama', 'Duitama', 'Tunja',
                 'BOGOTA', 'Tunja', 'duitama', 'Tunja ', 'bogota',
                 'Tunja', 'Duitama', 'tunja', 'Bogot\u00e1', 'DUITAMA',
                 'Bogota', 'Sogamoso', 'sogamoso', 'Bogota', 'Duitama ',
                 'tunja ', 'DUITAMA', 'Tunja'],
    'categoria':['Frutas', 'Lacteos', 'Abarrotes', 'Frutas', 'Lacteos',
                 'Frutas', 'Abarrotes', 'Lacteos', 'Abarrotes', 'Frutas',
                 'Lacteos', 'Abarrotes', 'Frutas', 'Lacteos', 'Abarrotes',
                 'Frutas', 'Lacteos', 'Abarrotes', 'Frutas', 'Lacteos',
                 'Abarrotes', 'Frutas', 'Lacteos', 'Abarrotes', 'Frutas',
                 'Lacteos', 'Lacteos', 'Lacteos'],
    'unidades': [3, 5, 2, 4, 1, 6, 3, 2, 4, 5,
                 2, 3, 1, 4, 6, 2, 3, 5, 4, 1,
                 2, 3, 4, -2, 5, 3, 1, 4],
    'precio':   [9000.0, 15000.0, np.nan, 10000.0, 16000.0,
                 8500.0, 980000.0, np.nan, 24000.0, 11000.0,
                 17000.0, 23000.0, 9500.0, 14000.0, 26000.0,
                 10500.0, 18000.0, np.nan, 12000.0, 15500.0,
                 45000.0, 9200.0, 16500.0, 25000.0, 8800.0,
                 17500.0, 15500.0, 14000.0],
    'calificacion': [5.0, 4.0, 3.0, 4.0, np.nan, 5.0, 2.0, 4.0, 9.0, 3.0,
                     4.0, 5.0, 2.0, 4.0, 3.0, 5.0, 4.0, 3.0, 4.0, 5.0,
                     4.0, 4.0, 3.0, 4.0, 5.0, np.nan, 5.0, 4.0],
})

print(ventas.shape)
print('duplicados exactos    :', ventas.duplicated().sum())
print('huecos por columna    :', ventas.isna().sum().to_dict())
print('ciudades distintas    :', ventas['ciudad'].nunique())
ventas.head(8)

### Las dos piezas de scikit-learn que la rubrica exige

La Actividad 2 nombra `KNNImputer` y los escaladores. No hay reto sobre ellos porque son
tres lineas de API, pero **tienen que aparecer en tu entrega**, asi que aqui quedan corriendo
para que los copies con confianza.

Dos advertencias que valen mas que la API:

1. `KNNImputer` mide distancias. Si en la tabla hay un 980000 sin corregir, ese vecino
   deforma todo. **Imputar con KNN antes de tratar outliers es un error de metodo.**
2. `fit_transform` sobre toda la tabla te filtra informacion del conjunto de prueba al de
   entrenamiento. Hoy no partimos datos, pero cuando lo hagas: `fit` en train, `transform`
   en test.

In [ ]:
try:
    from sklearn.impute import KNNImputer
    from sklearn.preprocessing import MinMaxScaler, StandardScaler
    import sklearn
    print('scikit-learn', sklearn.__version__)
except ImportError:
    raise SystemExit('Falta scikit-learn. Instala con: pip install scikit-learn')

# Trabajamos sobre una copia SIN el precio absurdo, para no envenenar al KNN.
base = ventas[ventas['precio'] != 980000.0].copy()
numericas = ['unidades', 'precio', 'calificacion']

knn = KNNImputer(n_neighbors=3)
imputado = pd.DataFrame(knn.fit_transform(base[numericas]),
                        columns=numericas, index=base.index)
print('huecos antes  :', int(base[numericas].isna().sum().sum()))
print('huecos despues:', int(imputado.isna().sum().sum()))

demo = base.copy()
demo['precio_minmax']   = MinMaxScaler().fit_transform(base[['precio']])
demo['precio_estandar'] = StandardScaler().fit_transform(base[['precio']])
print(demo[['precio', 'precio_minmax', 'precio_estandar']].describe().round(3))

Fijate en el `describe`: `MinMaxScaler` deja el minimo en 0 y el maximo en 1;
`StandardScaler` deja la media en 0 y la desviacion en 1. Ninguno de los dos arregla un
outlier: lo comprime, pero sigue estando en el borde. Escalar no es limpiar.

Ahora si, los retos.

---
## Reto 1 - Un detector de outliers con el metodo como parametro

En la clase normal se escriben dos funciones, una por criterio. Eso es un script. Aqui vas a
escribir **una sola funcion parametrizada** que devuelva una mascara booleana, que es lo que
se puede componer despues.

Escribe `detectar_outliers(df, columna, metodo='iqr', umbral=None)`.

**Contrato:**
- Devuelve una `pd.Series` de dtype `bool`, **con el mismo indice que `df`** y la misma
  longitud. `True` significa "esta fila es outlier en esa columna".
- `metodo='iqr'`: es outlier si el valor cae fuera de `[Q1 - k*IQR, Q3 + k*IQR]`, con los
  cuantiles de `Series.quantile` (interpolacion lineal, el default).
- `metodo='zscore'`: es outlier si `abs(x - media) / desviacion > k`, con `Series.std()`
  (desviacion **muestral**, `ddof=1`, el default de pandas).
- `umbral=None` significa "usa el valor de la casa": **1.5 para iqr y 3.0 para zscore**. Si
  te pasan un umbral explicito, manda ese.
- **Un hueco no es un outlier**: donde hay `NaN` la mascara vale `False`.
- **Dispersion cero**: si el `IQR` es 0 (metodo iqr) o la desviacion es 0 (metodo zscore),
  no hay criterio que aplicar. Devuelve todo `False`, sin dividir por cero, sin warnings y
  sin excepcion.
- Si `metodo` no es `'iqr'` ni `'zscore'`, lanza `ValueError` y **el mensaje debe contener
  el valor recibido**.
- Si la columna no es numerica, lanza `TypeError` y **el mensaje debe contener el nombre de
  la columna**. Usa `pd.api.types.is_numeric_dtype`.
- No modifica `df`.

In [ ]:
def detectar_outliers(df, columna, metodo='iqr', umbral=None):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 1 ---
_antes = ventas.copy()
m_iqr = detectar_outliers(ventas, 'precio')

assert isinstance(m_iqr, pd.Series), 'Debe devolver una pd.Series, no un array ni una lista'
assert m_iqr.dtype == bool, f'La mascara debe ser de dtype bool, llego {m_iqr.dtype}'
assert m_iqr.index.equals(ventas.index), \
    'La mascara debe conservar el indice de df: si no, no se puede usar para filtrar'
assert ventas.equals(_antes), 'detectar_outliers no debe modificar el DataFrame de entrada'

idx_extremo = ventas.index[ventas['precio'] == 980000.0][0]
idx_ambiguo = ventas.index[ventas['precio'] == 45000.0][0]

assert bool(m_iqr.loc[idx_extremo]), 'IQR debe marcar el precio de 980000'
assert bool(m_iqr.loc[idx_ambiguo]), 'IQR debe marcar tambien el precio de 45000'
assert int(m_iqr.sum()) == 2, f'Con IQR 1.5 hay exactamente 2 outliers, tu diste {int(m_iqr.sum())}'

m_z = detectar_outliers(ventas, 'precio', metodo='zscore')
assert bool(m_z.loc[idx_extremo]), 'Z-Score debe marcar el precio de 980000'
assert not bool(m_z.loc[idx_ambiguo]), \
    'Z-Score NO marca el 45000: el propio 980000 infla la desviacion y lo esconde. Ese es el punto del reto'
assert int(m_z.sum()) == 1, f'Con Z-Score 3.0 hay exactamente 1 outlier, tu diste {int(m_z.sum())}'

# los huecos nunca son outliers
assert not m_iqr[ventas['precio'].isna()].any(), 'Un NaN en precio no puede marcarse como outlier'
assert not m_z[ventas['precio'].isna()].any(), 'Un NaN en precio no puede marcarse como outlier'

# umbral por defecto: 1.5 para iqr, 3.0 para zscore
assert m_iqr.equals(detectar_outliers(ventas, 'precio', 'iqr', 1.5)), \
    'umbral=None en iqr debe comportarse igual que umbral=1.5'
assert m_z.equals(detectar_outliers(ventas, 'precio', 'zscore', 3.0)), \
    'umbral=None en zscore debe comportarse igual que umbral=3.0'
assert int(detectar_outliers(ventas, 'precio', 'zscore', 4.0).sum()) == 1, \
    'Con umbral 4.0 el 980000 (z = 4.80) sigue marcado'
assert int(detectar_outliers(ventas, 'precio', 'zscore', 5.0).sum()) == 0, \
    'Con umbral 5.0 ya no queda ningun outlier: el umbral explicito debe mandar'

# dispersion cero
constante = pd.DataFrame({'a': [5.0] * 8})
assert not detectar_outliers(constante, 'a', 'zscore').any(), \
    'Columna constante: desviacion 0, no hay outliers ni division por cero'
assert not detectar_outliers(constante, 'a', 'iqr').any(), \
    'Columna constante: IQR 0, no hay outliers'

casi = pd.DataFrame({'a': [7.0] * 10 + [99.0]})
assert not detectar_outliers(casi, 'a', 'iqr').any(), \
    'Q1 = Q3 = 7 y por lo tanto IQR = 0: sin dispersion no hay criterio, devuelve todo False'

solo_nan = pd.DataFrame({'a': [np.nan] * 6})
assert not detectar_outliers(solo_nan, 'a', 'iqr').any(), 'Columna toda NaN: todo False, sin reventar'
assert not detectar_outliers(solo_nan, 'a', 'zscore').any(), 'Columna toda NaN: todo False, sin reventar'

try:
    detectar_outliers(ventas, 'precio', metodo='mediana')
except ValueError as e:
    assert 'mediana' in str(e), 'El ValueError debe decir cual fue el metodo invalido que recibio'
else:
    raise AssertionError("Un metodo desconocido debe lanzar ValueError, no devolver algo")

try:
    detectar_outliers(ventas, 'vendedor')
except TypeError as e:
    assert 'vendedor' in str(e), 'El TypeError debe nombrar la columna no numerica'
else:
    raise AssertionError('Una columna de texto debe lanzar TypeError')

print('Reto 1 resuelto.')
print(f'IQR marca {int(m_iqr.sum())} filas | Z-Score marca {int(m_z.sum())}')

> **Si te sobra tiempo:** implementa un tercer metodo, `'mad'` (desviacion absoluta
> mediana): `0.6745 * abs(x - mediana) / mediana(abs(x - mediana)) > k`. Es la version
> robusta del Z-Score. Correlo sobre `precio` y compara: deberia marcar el 45000, que es
> justo lo que el Z-Score clasico no vio.

---
## Reto 2 - Normalizacion de texto que aguanta datos reales

`.str.strip().str.title()` deja `'Bogot\u00e1'` y `'Bogota'` como dos ciudades distintas, y
`'Bogota  D.C.'` como una tercera. Vamos a arreglarlo de verdad.

Escribe `normalizar_texto(df, columnas, sinonimos=None)`.

**Contrato:**
- Devuelve un **DataFrame nuevo**. No modifica el que recibe.
- Para cada nombre en `columnas` aplica, **en este orden**:
  1. quitar espacios de los extremos
  2. colapsar los espacios internos repetidos a uno solo (`'Bogota  D.C.'` -> `'Bogota D.C.'`)
  3. quitar tildes y diacriticos: `'Bogot\u00e1'` -> `'Bogota'`, `'Ni\u00f1o'` -> `'Nino'`
  4. dejarlo en formato titulo: `'bogota d.c.'` -> `'Bogota D.C.'`
  5. aplicar el diccionario `sinonimos`, cuyas **claves ya vienen en la forma normalizada**
     y cuyos valores se usan tal cual
- Los `NaN` se conservan como `NaN`. Ojo: `astype(str)` convierte el hueco en el texto
  `'nan'` y a partir de ahi ya no lo recuperas nunca.
- Si `sinonimos` es `None` se comporta como si fuera un diccionario vacio.
- Si alguna columna de `columnas` no existe, lanza `ValueError` con el **nombre de la
  columna** en el mensaje. No proceses las demas antes de fallar.
- Las columnas que no estan en `columnas` quedan intactas, y el orden de las columnas del
  DataFrame no cambia.

**Pista para el paso 3:** `unicodedata.normalize('NFKD', texto)` separa la `\u00e1` en `a`
mas una marca de acento; `unicodedata.combining(c)` te dice si `c` es una de esas marcas.
Ese par resuelve tildes, dieresis y la enie de una sola vez, sin tablas de reemplazo a mano.

In [ ]:
def normalizar_texto(df, columnas, sinonimos=None):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 2 ---
_antes = ventas.copy()
SINONIMOS = {'Bogota D.C.': 'Bogota'}
limpio = normalizar_texto(ventas, ['vendedor', 'ciudad'], SINONIMOS)

assert isinstance(limpio, pd.DataFrame), 'Debe devolver un DataFrame'
assert ventas.equals(_antes), 'normalizar_texto no debe modificar el DataFrame de entrada'
assert list(limpio.columns) == list(ventas.columns), 'El orden de las columnas no cambia'
assert len(limpio) == len(ventas), 'No se eliminan ni se agregan filas'

assert sorted(limpio['vendedor'].unique()) == ['Alvarez', 'Nino', 'Rojas', 'Suarez'], \
    f"Deben quedar 4 vendedores, tu dejaste {sorted(limpio['vendedor'].unique())}"
assert sorted(limpio['ciudad'].unique()) == ['Bogota', 'Duitama', 'Sogamoso', 'Tunja'], \
    f"Deben quedar 4 ciudades, tu dejaste {sorted(limpio['ciudad'].unique())}"

# el espacio doble se colapsa ANTES de buscar en el diccionario de sinonimos
sin_mapa = normalizar_texto(ventas, ['ciudad'])
assert 'Bogota D.C.' in set(sin_mapa['ciudad']), \
    "Sin sinonimos, 'Bogota  D.C.' debe quedar en 'Bogota D.C.' con UN solo espacio"
assert 'Bogota  D.C.' not in set(sin_mapa['ciudad']), 'El espacio interno doble debe colapsarse'
assert limpio['unidades'].equals(ventas['unidades']), 'Las columnas no listadas quedan intactas'

# tildes y enies
assert not any('\u00f1' in v or '\u00e1' in v for v in limpio['vendedor']), \
    'No debe quedar ninguna tilde ni enie en vendedor'

# los huecos sobreviven
mini = pd.DataFrame({'c': [' bOGOTA ', np.nan, 'Bogot\u00e1'], 'n': [1, 2, 3]})
r = normalizar_texto(mini, ['c'])
assert pd.isna(r.loc[1, 'c']), \
    "El NaN debe seguir siendo NaN: si te quedo el texto 'Nan' usaste astype(str)"
assert r.loc[0, 'c'] == 'Bogota' and r.loc[2, 'c'] == 'Bogota', 'Ambas variantes caen en Bogota'

# el diccionario de sinonimos se aplica al final, sobre el texto ya normalizado
r2 = normalizar_texto(mini, ['c'], {'Bogota': 'BOG'})
assert list(r2['c'].dropna()) == ['BOG', 'BOG'], \
    'Los sinonimos se aplican DESPUES de normalizar y su valor se usa tal cual'

try:
    normalizar_texto(ventas, ['ciudad', 'departamento'])
except ValueError as e:
    assert 'departamento' in str(e), 'El ValueError debe nombrar la columna que no existe'
else:
    raise AssertionError('Una columna inexistente debe lanzar ValueError')

# el premio gordo: el duplicado parcial solo aparece DESPUES de normalizar
assert ventas.duplicated().sum() == 1, 'En crudo solo se ve 1 duplicado exacto'
assert limpio.duplicated().sum() == 2, \
    'Tras normalizar el texto deben verse 2 duplicados: uno estaba escondido tras las mayusculas'

print('Reto 2 resuelto.')
print('duplicados visibles: antes', ventas.duplicated().sum(), '-> despues', limpio.duplicated().sum())

> **Si te sobra tiempo:** agrega el parametro `formato='titulo'` que acepte tambien
> `'minuscula'` y `'mayuscula'`. Y piensa el caso que rompe a `str.title()`: `"o'brien"`
> queda como `"O'Brien"`, pero `'d.c.'` queda como `'D.C.'` y `'mcdonald'` como
> `'Mcdonald'`. No hay solucion perfecta; hay que elegir y documentarla.

---
## Reto 3 - Imputar con la mediana de su grupo, no con la global

La mediana global del `precio` en esta tabla es 15500. Pero un hueco en Frutas no vale lo
mismo que un hueco en Abarrotes: sus medianas son 9500 y 25500. Rellenar los tres con 15500
inventa datos que ninguna categoria produjo.

Escribe `imputar_por_grupo(df, columna, por, estrategia='mediana')`.

**Contrato:**
- Devuelve un **DataFrame nuevo**, mismas columnas en el mismo orden, mismo indice, mismo
  numero de filas. No modifica el que recibe y **no agrega columnas auxiliares**.
- Rellena los `NaN` de `columna` con el estadistico de las filas que comparten el valor de
  `por` (un nombre de columna).
- Los valores que **no** eran `NaN` quedan identicos.
- `estrategia`: `'mediana'`, `'media'` o `'moda'`. En `'moda'`, si hay empate se toma la
  primera de `Series.mode()`. Cualquier otro valor lanza `ValueError` con **ese valor** en
  el mensaje.
- **Grupo sin datos:** si en un grupo todos los valores de `columna` son `NaN`, esas filas se
  rellenan con el estadistico **global** de la columna. Lo mismo para las filas cuyo valor de
  `por` sea `NaN`.
- **Columna sin datos:** si la columna entera es `NaN`, se devuelve tal cual, con sus `NaN`
  intactos y **sin lanzar excepcion**.
- El estadistico se calcula ignorando los `NaN` (comportamiento normal de pandas).

**Pista:** `df.groupby(por)[columna].transform(...)` devuelve una Serie con el mismo indice
que `df`, con el estadistico del grupo repetido en cada fila. Es exactamente lo que necesitas
para pasarselo a `fillna`, que empareja por indice.

In [ ]:
def imputar_por_grupo(df, columna, por, estrategia='mediana'):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 3 ---
_antes = ventas.copy()
imp = imputar_por_grupo(ventas, 'precio', 'categoria')

assert isinstance(imp, pd.DataFrame), 'Debe devolver un DataFrame'
assert ventas.equals(_antes), 'imputar_por_grupo no debe modificar el DataFrame de entrada'
assert list(imp.columns) == list(ventas.columns), 'Mismas columnas, mismo orden, sin columnas auxiliares'
assert imp.index.equals(ventas.index), 'Mismo indice'
assert imp['precio'].isna().sum() == 0, 'No debe quedar ningun hueco en precio'

huecos = ventas.index[ventas['precio'].isna()]
llenos = ventas.index[ventas['precio'].notna()]
assert imp.loc[llenos, 'precio'].equals(ventas.loc[llenos, 'precio']), \
    'Los valores que ya existian no se pueden tocar'

med_abarrotes = ventas.loc[ventas['categoria'] == 'Abarrotes', 'precio'].median()
med_lacteos   = ventas.loc[ventas['categoria'] == 'Lacteos', 'precio'].median()
med_global    = ventas['precio'].median()
assert med_abarrotes != med_global, 'Control del ejercicio: la mediana del grupo si difiere de la global'

for i in huecos:
    esperado = med_abarrotes if ventas.loc[i, 'categoria'] == 'Abarrotes' else med_lacteos
    assert imp.loc[i, 'precio'] == esperado, (
        f"Fila {i} ({ventas.loc[i, 'categoria']}): esperaba la mediana de su categoria "
        f"({esperado}) y llego {imp.loc[i, 'precio']}. Si te dio {med_global} usaste la mediana global")

# la media es otra cosa: en Abarrotes el 980000 la dispara
imp_media = imputar_por_grupo(ventas, 'precio', 'categoria', estrategia='media')
media_abarrotes = ventas.loc[ventas['categoria'] == 'Abarrotes', 'precio'].mean()
i_ab = [i for i in huecos if ventas.loc[i, 'categoria'] == 'Abarrotes'][0]
assert abs(imp_media.loc[i_ab, 'precio'] - media_abarrotes) < 1e-6, \
    "estrategia='media' debe usar la media del grupo"
assert imp_media.loc[i_ab, 'precio'] > 100000, \
    'La media de Abarrotes va por encima de 100000 porque el outlier sigue ahi: por eso se limpia antes de imputar'

# grupo entero vacio y clave de grupo NaN -> estadistico global
mini = pd.DataFrame({'g': ['A', 'A', 'B', 'B', 'C', None],
                     'v': [1.0, 3.0, np.nan, np.nan, 10.0, np.nan]})
r = imputar_por_grupo(mini, 'v', 'g')
assert r.loc[2, 'v'] == 3.0 and r.loc[3, 'v'] == 3.0, \
    'El grupo B no tiene ningun dato: se rellena con la mediana global (3.0), no con NaN'
assert r.loc[5, 'v'] == 3.0, 'Una fila con la clave de grupo en NaN tambien cae en el global'
assert r.loc[0, 'v'] == 1.0 and r.loc[4, 'v'] == 10.0, 'Los valores existentes no se tocan'

# columna entera vacia: se devuelve como esta, sin reventar
vacia = pd.DataFrame({'g': ['A', 'B'], 'v': [np.nan, np.nan]})
rv = imputar_por_grupo(vacia, 'v', 'g')
assert rv['v'].isna().all(), 'Sin ningun dato no hay nada con que rellenar: los NaN se quedan'

# la moda
modas = pd.DataFrame({'g': ['A', 'A', 'A', 'B'], 'v': [2.0, 2.0, np.nan, np.nan]})
rm = imputar_por_grupo(modas, 'v', 'g', estrategia='moda')
assert rm.loc[2, 'v'] == 2.0, "estrategia='moda' rellena con el valor mas frecuente del grupo"
assert rm.loc[3, 'v'] == 2.0, 'El grupo B esta vacio: cae en la moda global'

try:
    imputar_por_grupo(ventas, 'precio', 'categoria', estrategia='promedio')
except ValueError as e:
    assert 'promedio' in str(e), 'El ValueError debe decir cual estrategia invalida recibio'
else:
    raise AssertionError("estrategia='promedio' debe lanzar ValueError")

print('Reto 3 resuelto.')
print(f'mediana global {med_global:,.0f} | Frutas '
      f"{ventas.loc[ventas['categoria'] == 'Frutas', 'precio'].median():,.0f} | "
      f'Abarrotes {med_abarrotes:,.0f}')

> **Si te sobra tiempo:** agrega `estrategia='knn'` usando `KNNImputer` **dentro de cada
> grupo**, con las demas columnas numericas como vecinas. Vas a chocar con dos cosas reales:
> los grupos con menos filas que `n_neighbors`, y que el imputador devuelve un array sin
> indice. Resolver eso es exactamente lo que separa una funcion de juguete de una usable.

---
## Reto 4 - La herramienta: pipeline componible y validacion por contrato

Este es el reto largo. Son dos funciones y ninguna toca los datos directamente: **operan
sobre otras funciones y sobre reglas**. Ahi esta la diferencia entre un notebook que limpio
una tabla y una herramienta que limpia las que vengan.

### 4A - `aplicar_pipeline(df, pasos)`

**Contrato:**
- `pasos` es una lista de tuplas `(nombre, funcion)`, donde `funcion` recibe un DataFrame y
  devuelve un DataFrame. Se aplican **en orden**: cada una recibe la salida de la anterior.
- Devuelve la tupla `(df_final, bitacora)`.
- `bitacora` es una **lista de diccionarios**, uno por paso, en el mismo orden, con
  exactamente estas seis claves:

| Clave | Valor |
|---|---|
| `'paso'` | el nombre que venia en la tupla |
| `'filas_antes'` | `len` del DataFrame antes del paso (int) |
| `'filas_despues'` | `len` despues (int) |
| `'nulos_antes'` | total de `NaN` en **toda** la tabla antes (int) |
| `'nulos_despues'` | total de `NaN` en toda la tabla despues (int) |
| `'error'` | `None` si el paso corrio bien; si no, el mensaje de la excepcion (str no vacio) |

- **Un paso que revienta no tumba el pipeline.** Se atrapa la excepcion, se anota en
  `'error'`, el DataFrame **queda como estaba antes de ese paso** (por lo tanto
  `filas_despues == filas_antes` y `nulos_despues == nulos_antes`) y se sigue con el
  siguiente paso.
- Si un paso devuelve algo que no es un DataFrame, cuenta como error igual.
- `pasos` vacia: devuelve una copia de `df` y `bitacora == []`.
- No modifica el `df` original ni siquiera cuando los pasos si modifican lo que reciben.

### 4B - `validar_contrato(df, reglas)`

**Contrato:**
- `reglas` es un diccionario que puede traer estas tres claves, todas opcionales:
  - `'obligatorias'`: lista de columnas que deben existir
  - `'tipos'`: dict `{columna: 'numerico' | 'texto'}`
  - `'rangos'`: dict `{columna: (minimo, maximo)}`, ambos extremos **incluidos**
- Devuelve una **lista de diccionarios**: todas las violaciones encontradas. **No se detiene
  en la primera y no lanza excepcion por los datos.** Si no hay ninguna, lista vacia.
- Cada violacion tiene exactamente estas cuatro claves:

| Clave | Valor |
|---|---|
| `'regla'` | `'columna_faltante'`, `'tipo_incorrecto'` o `'fuera_de_rango'` |
| `'columna'` | nombre de la columna |
| `'filas'` | lista de etiquetas de indice de las filas culpables, ascendente. Lista **vacia** en las reglas que son de la columna entera |
| `'mensaje'` | texto legible que **debe contener el nombre de la columna** |

- Una columna mencionada en cualquiera de las tres reglas y que no exista produce **una sola**
  violacion `'columna_faltante'` (aunque aparezca en las tres) y no se le hacen mas chequeos.
- `'numerico'` se comprueba con `pd.api.types.is_numeric_dtype`; `'texto'` es su negacion.
- Un rango solo se evalua si la columna existe y es numerica. Si existe pero no es numerica,
  se salta en silencio: de eso ya se encarga `'tipos'`.
- Un `NaN` **no viola** un rango.
- La lista sale **ordenada por `(regla, columna)`**, para que el informe sea reproducible.
- Si algun valor de `'tipos'` no es `'numerico'` ni `'texto'`, ahi si `raise ValueError`: eso
  no es un dato sucio, es un contrato mal escrito.

In [ ]:
def aplicar_pipeline(df, pasos):
    # tu codigo aqui
    pass


def validar_contrato(df, reglas):
    # tu codigo aqui
    pass


In [ ]:
# --- verificacion del reto 4 ---
_antes = ventas.copy()

PASOS = [
    ('normalizar', lambda d: normalizar_texto(d, ['vendedor', 'ciudad'], {'Bogota D.C.': 'Bogota'})),
    ('duplicados', lambda d: d.drop_duplicates(keep='first').reset_index(drop=True)),
    ('outliers',   lambda d: d[~detectar_outliers(d, 'precio', 'zscore')].reset_index(drop=True)),
    ('imputar',    lambda d: imputar_por_grupo(d, 'precio', 'categoria')),
    ('paso_roto',  lambda d: d['columna_que_no_existe']),
]
final, bitacora = aplicar_pipeline(ventas, PASOS)

assert isinstance(final, pd.DataFrame), 'El primer elemento devuelto es un DataFrame'
assert isinstance(bitacora, list), 'El segundo elemento devuelto es una lista'
assert ventas.equals(_antes), 'aplicar_pipeline no debe modificar el DataFrame original'
assert len(bitacora) == len(PASOS), f'Un registro por paso: esperaba {len(PASOS)}, llegaron {len(bitacora)}'
assert [b['paso'] for b in bitacora] == [n for n, _ in PASOS], \
    'La bitacora va en el mismo orden de los pasos y con los mismos nombres'
CLAVES = {'paso', 'filas_antes', 'filas_despues', 'nulos_antes', 'nulos_despues', 'error'}
for b in bitacora:
    assert set(b) == CLAVES, f"El registro '{b.get('paso')}' debe traer exactamente {sorted(CLAVES)}"

assert bitacora[0]['filas_antes'] == 28 and bitacora[0]['filas_despues'] == 28, \
    'Normalizar texto no cambia el numero de filas'
assert bitacora[1]['filas_despues'] == 26, \
    'Tras normalizar, drop_duplicates debe quitar 2 filas (28 -> 26), no 1'
assert bitacora[2]['filas_despues'] == 25, 'Z-Score saca la fila del precio de 980000 (26 -> 25)'
assert bitacora[3]['nulos_despues'] < bitacora[3]['nulos_antes'], 'Imputar tiene que bajar la cuenta de nulos'
assert bitacora[3]['filas_despues'] == bitacora[3]['filas_antes'], 'Imputar no elimina filas'

# el eslabon encadenado: lo que entra a un paso es lo que salio del anterior
for previo, actual in zip(bitacora, bitacora[1:]):
    assert actual['filas_antes'] == previo['filas_despues'], \
        f"El paso '{actual['paso']}' debe recibir la salida de '{previo['paso']}', no el df original"

# el paso roto se anota y NO tumba el pipeline
roto = bitacora[-1]
assert isinstance(roto['error'], str) and roto['error'], \
    'Un paso que revienta debe dejar el mensaje de la excepcion en la clave error, no None'
assert roto['filas_despues'] == roto['filas_antes'], 'Si el paso fallo, la tabla queda como estaba'
assert roto['nulos_despues'] == roto['nulos_antes'], 'Si el paso fallo, la tabla queda como estaba'
assert all(b['error'] is None for b in bitacora[:-1]), 'Los pasos que corrieron bien llevan error=None'
assert len(final) == 25, 'Aunque el ultimo paso fallo, el resultado es el del paso 4'
assert final['precio'].isna().sum() == 0, 'El pipeline completo deja precio sin huecos'

no_es_df = aplicar_pipeline(ventas, [('mal', lambda d: 'esto no es un DataFrame')])[1][0]
assert isinstance(no_es_df['error'], str) and no_es_df['error'], \
    'Un paso que devuelve algo que no es DataFrame cuenta como error'

f0, b0 = aplicar_pipeline(ventas, [])
assert b0 == [] and f0.equals(ventas), 'Sin pasos: copia del df y bitacora vacia'
assert f0 is not ventas, 'Aun sin pasos debe devolver una copia, no el mismo objeto'

# --- 4B ---
REGLAS = {
    'obligatorias': ['id_venta', 'vendedor', 'precio', 'descuento'],
    'tipos': {'precio': 'numerico', 'vendedor': 'texto', 'unidades': 'texto', 'descuento': 'numerico'},
    'rangos': {'calificacion': (1, 5), 'unidades': (1, 100), 'precio': (0, 500000)},
}
informe = validar_contrato(ventas, REGLAS)

assert isinstance(informe, list) and all(isinstance(v, dict) for v in informe), \
    'Debe devolver una lista de diccionarios'
for v in informe:
    assert set(v) == {'regla', 'columna', 'filas', 'mensaje'}, \
        f'Cada violacion lleva exactamente 4 claves, llego {sorted(v)}'
    assert v['columna'] in v['mensaje'], 'El mensaje debe nombrar la columna afectada'
    assert isinstance(v['filas'], list), 'filas siempre es una lista, aunque este vacia'

resumen = sorted((v['regla'], v['columna']) for v in informe)
assert resumen == [
    ('columna_faltante', 'descuento'),
    ('fuera_de_rango', 'calificacion'),
    ('fuera_de_rango', 'precio'),
    ('fuera_de_rango', 'unidades'),
    ('tipo_incorrecto', 'unidades'),
], f'El informe esperado tiene 5 violaciones; tu diste {resumen}'

assert informe == sorted(informe, key=lambda v: (v['regla'], v['columna'])), \
    'La lista sale ordenada por (regla, columna)'

por_clave = {(v['regla'], v['columna']): v for v in informe}
idx_calif = list(ventas.index[ventas['calificacion'] == 9.0])
idx_unid  = list(ventas.index[ventas['unidades'] < 0])
assert por_clave[('fuera_de_rango', 'calificacion')]['filas'] == idx_calif, \
    'fuera_de_rango debe reportar las etiquetas de indice de las filas culpables'
assert por_clave[('fuera_de_rango', 'unidades')]['filas'] == idx_unid, 'Las unidades en -2 violan el rango'
assert por_clave[('columna_faltante', 'descuento')]['filas'] == [], \
    'Una regla de columna entera lleva filas = []'
assert sum(1 for v in informe if v['regla'] == 'columna_faltante') == 1, \
    "'descuento' aparece en obligatorias Y en tipos, pero se reporta UNA sola vez"
assert ('tipo_incorrecto', 'descuento') not in por_clave, \
    'A una columna que no existe no se le revisa el tipo'
assert not any(v['columna'] == 'calificacion' and v['regla'] == 'columna_faltante' for v in informe), \
    'calificacion si existe: no puede aparecer como faltante'

# un NaN no viola un rango
solo_hueco = pd.DataFrame({'x': [1.0, np.nan, 3.0]})
assert validar_contrato(solo_hueco, {'rangos': {'x': (0, 10)}}) == [], \
    'Un NaN no es una violacion de rango: la mascara booleana tiene que cerrarse con fillna(False)'

assert validar_contrato(ventas, {}) == [], 'Sin reglas no hay violaciones'
assert validar_contrato(ventas, {'rangos': {'precio': (0, 1000000)}}) == [], \
    'Con un rango amplio, precio no viola nada'

# un rango sobre columna de texto se salta en silencio
assert validar_contrato(ventas, {'rangos': {'vendedor': (0, 5)}}) == [], \
    'Un rango sobre una columna no numerica se ignora: de eso se encarga la regla de tipos'

try:
    validar_contrato(ventas, {'tipos': {'precio': 'entero'}})
except ValueError as e:
    assert 'precio' in str(e) or 'entero' in str(e), \
        'El ValueError debe decir que tipo o que columna esta mal declarada'
else:
    raise AssertionError("Un tipo declarado que no es 'numerico' ni 'texto' debe lanzar ValueError")

print('Reto 4 resuelto.\n')
print(f"{'paso':<12}{'filas':>12}{'nulos':>12}   error")
for b in bitacora:
    print(f"{b['paso']:<12}{b['filas_antes']:>5} -> {b['filas_despues']:<4}"
          f"{b['nulos_antes']:>5} -> {b['nulos_despues']:<4}   {b['error'] or '-'}")
print()
for v in informe:
    print(f"  [{v['regla']:<17}] {v['mensaje']} {v['filas'] if v['filas'] else ''}")

> **Si te sobra tiempo:** haz que `aplicar_pipeline` acepte un parametro
> `detener_en_error=False`. Y despues piensa cual de los dos comportamientos quieres por
> defecto en produccion: seguir y entregar una tabla a medio limpiar, o parar en seco. No hay
> respuesta universal, pero si hay una respuesta para cada caso de uso, y hay que escribirla
> en el docstring.

---
## Reto abierto - sin verificacion automatica

Los cuatro retos anteriores tenian una respuesta correcta. Este no.

**El encargo:** el dueno de la tienda te va a entregar el archivo del mes que viene y quiere
"que quede limpio". Antes de tocar nada, te toca decidir **que se puede corregir en silencio
y que hay que preguntarle a el**. Ese es el trabajo real: la parte tecnica ya la resolviste.

Escribe `informe_calidad(df)` que imprima un reporte corto dirigido a **alguien que no
programa**, distinguiendo dos bloques:

- **Corregido automaticamente**: lo que arreglaste con criterio defendible, con cuantas
  filas toco cada correccion.
- **Requiere decision humana**: lo que NO tocaste porque no te corresponde decidirlo, con la
  pregunta concreta que le harias.

Preguntas que vale la pena que el informe conteste, y no todas se contestan igual:

- El precio de 980000: dedo pegado al teclado, venta mayorista real, o precio en otra moneda.
  Borrarlo, corregirlo a 9800 o dejarlo cambia el analisis. Cual eliges y con que argumento.
- El de 45000: IQR lo marca, Z-Score no. Con dos criterios que se contradicen, cual mandas.
- Los dos duplicados: uno es la misma venta grabada dos veces; el otro comparte `id_venta`
  pero es una fila distinta. Merecen el mismo tratamiento.
- Una calificacion de 9 en escala de 1 a 5: la vuelves 5, la vuelves `NaN`, o botas la fila.
- Cuantas filas quedan al final y **que porcentaje de la tabla original es informacion
  inventada por ti** al imputar. Ese numero deberia ir en todo informe de limpieza y casi
  nunca va.

Se juzga el **criterio**, no la salida: que decidiste, por que, y si el informe le sirve a
alguien que no sabe programar. Puedes reutilizar tus cuatro funciones.

In [ ]:
def informe_calidad(df):
    # tu codigo aqui
    pass


informe_calidad(ventas)

**Tus decisiones de diseno** (escribe aqui que corregiste solo y que dejaste en manos
del dueno, con el argumento de cada una):

1.
2.
3.

---
## Como se conecta con la Actividad 2

La **Actividad 2 (25%)** se entrega en **`.ipynb` + PDF**. Pide,
con estas palabras:

| Lo que pide la actividad | Donde quedo hoy |
|---|---|
| Dataset publico en CSV (Kaggle, datos.gov.co) cargado con pandas | Te falta: elige el archivo |
| Imputacion de faltantes: media, mediana o **KNN** | Reto 3 (media/mediana/moda por grupo) y la celda de `KNNImputer` |
| Duplicados | Reto 2 (los descubre) y Reto 4 (los quita dentro del pipeline) |
| Valores fuera de rango | Reto 4B, `validar_contrato` |
| Outliers con **Z-Score o IQR** | Reto 1, con los dos criterios y la discusion de cuando difieren |
| Normalizar o escalar con **MinMaxScaler o StandardScaler** | La celda de scikit-learn de arriba |
| **Funciones reutilizables para cada paso** | Retos 1 a 4: es literalmente el criterio de rubrica |
| Codigo comentado y estructurado | Docstrings y contratos, al armar la entrega |

Lo que hiciste aqui **excede** lo que pide la rubrica en tres puntos, y conviene que se note
en la entrega:

1. Tus funciones estan **parametrizadas** (`metodo`, `estrategia`, `umbral`), no duplicadas
   una por caso.
2. Tienes una **bitacora** que documenta cuantas filas toco cada paso. Eso responde solo la
   pregunta "explique su proceso de limpieza" que aparece en toda rubrica.
3. Tienes **validacion por contrato**, que es la unica forma de saber si la limpieza quedo
   bien sin revisar la tabla a ojo.

**Lo que te falta para entregar**, y es poco:

- Bajar un CSV publico de verdad y cargarlo con `pd.read_csv`. Ojo con `encoding` y con
  `sep=';'`, que en los datos abiertos colombianos aparece mas de lo que uno quisiera.
- Correr tu pipeline sobre ese CSV. Si tus funciones estan bien escritas, **no deberias
  cambiarles una linea**: solo la lista de pasos y las reglas. Si te toca reescribirlas,
  ahi tienes la leccion.
- Aplicar un escalador a una variable numerica y decir **por que** a esa y no a otra.
- Exportar el PDF del notebook con las salidas visibles.

Un consejo que vale nota: no entregues solo la tabla limpia. Entrega el **antes y despues**
(filas, nulos, categorias unicas) y una frase explicando cada decision. La rubrica premia el
proceso, y el proceso ya lo tienes escrito en la bitacora.

---
## Para terminar

1. Ejecuta el notebook completo de arriba abajo y confirma que las cuatro verificaciones pasan.
2. Guardalo **con las salidas visibles** (Archivo > Guardar).
3. Si quieres, subelo al aula (opcional, no se califica) y lo reviso y te comento por escrito.

Si terminaste temprano: junta tus cinco funciones en un archivo `limpieza.py` y hazle
`from limpieza import *` al notebook. Vas a descubrir de una que las funciones que dependen
de variables globales no se dejan mover. Ese ejercicio es el que te va a ahorrar dias en el
proyecto de la Seccion 3, que se entrega en `.py` (ademas del `.ipynb`).

**Proxima sesion:** visualizacion con matplotlib y seaborn. La via avanzada arranca donde
esta termina.

*Dudas: consulta al profesor*